<a href="https://colab.research.google.com/github/BryanStats/T-Mobile_Tuedays_Deals_Analysis-NLP-/blob/main/Final_Project_tmobile_tuesdays_lda_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Final Project: Brand Identification(T-Mobile Tuesday) Using Topic Modeling**
## By Jennifer Mac & Bryan Ramirez

For our final project, we will be following the 25th Top NLP Project, "Topic Modeling Using Latent Dirichlet Allocation (LDA)". In this project we will be using data from T-Mobile, an American wireless network operator. One of the notable perks for getting a T-Mobile plan is that every Tuesday, members get discounts on 3 to 7 random items, from cheap food discounts to high end clothing brands like Adidas.

Since T-Mobile has is a pricier mobile plan, we wanted to see if it's actually worth it by categorizing the discounted items into groups (Using Topic Modeling) and seeing what the break down of discounted items were. An example of these categories would be something like food, gas, clothes, cosmetics, etc. This would allow us to see what percentage of these discounts an individual would actually use. An Example would be if food makes up 60% of all discount codes and clothes only make up 5% of codes, we would expect an an individual who only uses food and clothes discounts to have the potential to use up to 65% of discount codes given.

To gather this data, we were scrapping from a forum website called Reddit which contains a sub forum called r/TMobileTuesdays. This subforum has threads for each week that contain comments of people sending their unused codes and requesting codes going from 2016 to present day. We used a third-party search tool designed for deep-dive archiving and research of Reddit data called Arctic Shift (arctic-shift.photon-reddit.com) to scrape this data.

# Step 1 – Setup

Fetch posts from the r/TMobileTuesdays subreddit using the Arctic Shift API (commented out - API currently unavailable).

In [ ]:
!pip install thefuzz python-Levenshtein
!pip install sentence-transformers
import re
import time
import requests
import numpy as np
import pandas as pd
from thefuzz import fuzz
from thefuzz import process
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

For each post, paginate through and fetch all top-level comments using the Arctic Shift comments API (commented out - API currently unavailable).

In [ ]:
# # Step 1: Get posts (your working code)
# url = 'https://arctic-shift.photon-reddit.com/api/posts/search'
# params = {'subreddit': 'TMobileTuesdays', 'limit': 100}
# response = requests.get(url, params=params)
# data = response.json()

Clean and normalize the scraped comments: lowercase, remove unwanted characters, and standardize brand name variations (commented out`- used in development).

In [ ]:
# # Step 2: For each post, fetch its comments
# all_data = []

# for post in data['data']:
#     post_id = post.get('id')
#     title = post.get('title', '')[45:]
#     num = post.get('num_comments', 0)
#     comments_url = 'https://arctic-shift.photon-reddit.com/api/comments/search'

#     all_comments = []
#     before = None
#     while True:
#         comments_params = {'link_id': post_id,
#                            'limit': 100,}
#         if before:
#             comments_params['before'] = before
#         comments_response = requests.get(comments_url, params=comments_params)

#         if comments_response.status_code != 200:
#             break
#         comments_data = comments_response.json()

#         if not comments_data or not comments_data.get('data'):
#             break
#         batch = comments_data['data']
#         all_comments.extend(batch)

#         print(f"  Fetched {len(all_comments)} comments so far...")

#         if len(batch) < 100:
#             break

#         # Use oldest comment's created_utc as cursor
#         before = min(c['created_utc'] for c in batch)
#         time.sleep(0.3)

#     # Only keep top-level comments
#     comments = [c.get('body', '')
#                 for c in all_comments
#                 if c.get('parent_id', '').startswith('t3_')]

#     all_data.append({
#         'date': title,
#         'num_com': num,
#         'comments': comments
#     })

Step 2 header - Reddit scraping section.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_excel('/content/drive/MyDrive/Stat_653/project/tmobiletuesdays_raw.xlsx')

import ast

def parse_comments(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)  # converts "['a', 'b']" → ['a', 'b']
            if isinstance(parsed, list):
                return parsed
        except (ValueError, SyntaxError):
            return [x]  # plain string, not a list-string
    return x

df['comments'] = df['comments'].apply(parse_comments)

In [ ]:
to_delete = [
    'codes', 'code', 'please', 'cb', 'hey dude', 'bogo', 'for', 'that', 'wants',
    'heres', 'the', 'deep dish', 'trade', 'https', 'reply', 'if', 'used', 'someone',
    'can', 'share', 'last', 'minute', 'cheese', 'italian', 'crazy puffs', 'crazy combo',
    'have', 'want', 'wild', 'bread', 'jumbo jack', 'heydude', 'dining', 'sticks'
]

to_remove = ['movie ticket', 'movie', 'atom', 'mj', 'cesars', 'little cesars', 'little ceasars',
             'lc', 'little c', 'gas', 'flowers', 'ink card', 'bk', 'croc', 'panda express',
             'panda', 'fh subs', 'fh', 'emags', 'magscom', 'mags', 'magazine', 'heydude',
             'firehouse subs', 'firehouse', 'tropical smoothie', 'smoothie cafe', 'smoothie',
             'sams club', 'sams', 'tropical cafe', 'freeprints gifts', 'freeprints',
             'jumbo', 'taco', 'pizza melt', 'advantage', 'popcorn', 'jack in box', 'jib', 'jb']

to_replace = ['movie ticket', 'movie ticket', 'movie ticket', 'movie ticket', 'little caesars',
              'little caesars', 'little caesars', 'little caesars', 'little caesars', 'shell',
              'bouqs', 'ink cards', 'burger king', 'crocs', 'panda express', 'panda express',
              'firehouse subs', 'firehouse subs', 'magazine', 'magazine', 'magazine', 'magazine', 'hey dude',
              'firehouse subs', 'firehouse subs', 'tropical smoothie cafe', 'tropical smoothie cafe', 'tropical smoothie cafe',
              'sams club', 'sams club', 'tropical smoothie cafe', 'freeprints', 'freeprints',
              '', '', '', '', '', 'jack in the box', 'jack in the box', 'jack in the box']

def clean_comment(c):
    c = re.sub(r'https?\S+', '', c)                    # remove URLs
    c = re.sub(r'[^\w\s:=]', '', c)                    # remove punctuation (keep : and =)
    for word in to_delete:
        c = re.sub(rf'\b{re.escape(word)}\b', '', c, flags=re.IGNORECASE)
    for i, word in enumerate(to_remove):
        c = re.sub(rf'\b{re.escape(word)}\b', to_replace[i], c, flags=re.IGNORECASE)
    c = re.sub(r' +', ' ', c).strip()                  # collapse extra spaces
    c = re.sub(r'\b(\w+)(\s+\1)+\b', r'\1', c, flags=re.IGNORECASE)  # remove repeated words
    return c

df['comments'] = df['comments'].apply(
    lambda x: [clean_comment(c) for c in x if isinstance(c, str)]
)
df

# Step 2 – Scrape posts/comments

Apply semantic similarity using sentence transformers and normalize known brand aliases to group brands both by fuzzy string matching and semantic meaning (commented out - used in development).

In [ ]:
# from sentence_transformers import SentenceTransformer
# from sklearn.metrics.pairwise import cosine_similarity
# import numpy as np
# from thefuzz import fuzz
# from thefuzz import process

model = SentenceTransformer('all-MiniLM-L6-v2')

def group_similar_brands_semantic(brand_counts, fuzzy_threshold=75, semantic_threshold=0.6):
    canonical = {}
    grouped = Counter()

    # No normalization needed - dataset already cleaned
    sorted_brands = sorted(brand_counts.items(), key=lambda x: x[1], reverse=True)
    brand_names = [b[0] for b in sorted_brands]

    if not brand_names:
        return grouped

    embeddings = model.encode(brand_names)

    for idx, (brand, count) in enumerate(sorted_brands):
        matched = False

        for canonical_brand in list(canonical.keys()):
            fuzzy_score = fuzz.token_sort_ratio(brand, canonical_brand)
            canonical_idx = brand_names.index(canonical_brand)
            semantic_score = cosine_similarity(
                [embeddings[idx]],
                [embeddings[canonical_idx]]
            )[0][0]

            if fuzzy_score >= fuzzy_threshold or semantic_score >= semantic_threshold:
                canonical[brand] = canonical[canonical_brand]
                matched = True
                break

        if not matched:
            canonical[brand] = brand

    for brand, count in brand_counts.items():
        grouped[canonical[brand]] += count

    return grouped

Extract brand names from comments using regex patterns to identify lines in 'Brand: CODE' format, then apply fuzzy matching to group similar brand names (commented out - used in development).

In [ ]:
def extract_brands(comments):
    brands = []
    patterns = [
        r'^([A-Za-z][A-Za-z\s\'\-\/\$\.\(\)&]+)[=:]\s*[A-Za-z0-9]{4,}',
        r'^([A-Za-z][A-Za-z\s\'\-\/\$\.\(\)&]+)\s+[A-Z0-9]{6,}',
    ]
    for comment in comments:
        if not isinstance(comment, str):
            continue
        for line in comment.split('\n'):
            line = line.strip()
            if len(line.split()) > 4:
                continue
            for pattern in patterns:
                match = re.match(pattern, line)
                if match:
                    brands.append(match.group(1).strip().lower())
                    break
    return brands


# Applied to each row
def top_brands_per_row(comments, top_n=10):
    brands = extract_brands(comments)
    brand_counts = Counter(brands)
    grouped = group_similar_brands_semantic(brand_counts)
    return [(brand, count) for brand, count in grouped.most_common(top_n) if count > 2]

df['top_brands'] = df['comments'].apply(lambda x: top_brands_per_row(x))
df[['date', 'top_brands']]
df = df.drop(columns=['comments']) # drops comment row
df = df[df['top_brands'].map(len) > 0] # drop rows with no discounted items

In [ ]:
# Explode top_brands into separate rows
df_exploded = df.copy()
df_exploded = df_exploded[df_exploded['top_brands'].apply(len) > 0]  # remove empty rows

# Create separate columns for brand and count
df_exploded['brand'] = df_exploded['top_brands'].apply(lambda x: [(b, c) for b, c in x])
df_exploded = df_exploded.explode('brand')

# Split brand tuple into separate columns
df_exploded['count'] = df_exploded['brand'].apply(lambda x: x[1])
df_exploded['brand'] = df_exploded['brand'].apply(lambda x: x[0])

# Clean up top_brands column to only show brand names without counts
df_exploded['top_brands'] = df_exploded['top_brands'].apply(lambda x: [b for b, c in x])

df_exploded = df_exploded.reset_index(drop=True)
df_exploded = df_exploded.rename(columns={
    'top_brands': 'item_list',
    'brand': 'discount_item'
})

df_exploded

Save the final dataframe to Google Drive as an Excel file for future use (commented out - used in development).

In [ ]:
brand_descriptions = {
    'shell': 'Gas fuel station fill up tank',
    'little caesars': 'Pizza hot ready budget friendly takeout',
    'pizza hut': 'Pizza pan stuffed crust sit down delivery',
    'pizza': 'Pizza slice discount takeout delivery',
    'kfc': 'Fried chicken crispy bucket combo meal',
    'popeyes': 'Cajun fried chicken spicy sandwich biscuit',
    'wingstop': 'Chicken wings flavors lemon pepper combo',
    'burger king': 'Whopper flame grilled burger fast food',
    'wendys': 'Square burger frosty baconator fast food',
    'shake shack': 'Premium smash burger milkshake crinkle fries',
    'jack in box': 'Tacos burger sourdough jack drive thru',
    'jack': 'Tacos burger sourdough jack drive thru',
    'sonic': 'Drive in slush hotdog corn dog carhop',
    'panera': 'Soup bread bowl sandwich bakery cafe',
    'qdoba': 'Burrito bowl queso mexican grilled chicken',
    'panda express': 'Orange chicken fried rice chinese american combo',
    'firehouse subs': 'Hot submarine sandwich hook ladder hero',
    'lazy dog': 'Comfort food sit down casual american dining',
    'blue dolphin': 'Casual sit down dining restaurant meal',
    'kung fu tea': 'Boba bubble tea taro milk drink',
    'tropical smoothie cafe': 'Blended smoothie acai flatbread cafe snack',
    'hello fresh': 'Weekly meal kit recipe ingredients home cooking',
    'factor': 'Pre made ready eat healthy meal delivery',
    'home chef': 'Meal kit recipe box fresh ingredients cooking',
    'cvs': 'Pharmacy prescription refill medicine cabinet health wellness vitamins supplements over counter cold flu allergy pain relief bandage first aid beauty skincare haircare cosmetics drugstore neighborhood convenience',
    'walgreens': 'Pharmacy prescription refill medicine cabinet health wellness vitamins supplements over counter cold flu allergy pain relief bandage first aid beauty skincare haircare cosmetics drugstore neighborhood convenience',
    'movie ticket': 'Cinema movie theater ticket discount screening',
    'amc': 'Cinema movie theater ticket discount screening',
    'legoland': 'LEGO theme park rides family kids amusement',
    'mgm': 'Casino hotel resort rewards loyalty points',
    'webtoon': 'Webcomic digital comics episodes manhwa series',
    'smurfs': 'Mobile game cartoon characters adventure kids',
    'funko': 'Vinyl pop figure collectible characters pop culture',
    'wsj wine': 'Wine club subscription curated bottles delivery',
    'costco': 'Wholesale warehouse bulk membership shopping savings',
    'sams club': 'Wholesale warehouse bulk membership shopping savings',
    'parking spot': 'Airport parking garage shuttle reservation spot',
    'adidas': 'Sneakers running shoes athletic training sportswear',
    'reebok': 'Sneakers cross training shoes classic athletic',
    'puma': 'Sneakers lifestyle shoes athletic streetwear fashion',
    'nobull': 'Training shoes cross fit performance athletic workout',
    'all birds': 'Wool sustainable eco friendly sneakers comfortable',
    'crocs': 'Foam clog sandal slip on casual comfort',
    'new era': 'Fitted cap snapback hat MLB team logo',
    'foco': 'Fan gear bobblehead team jersey sports collectible',
    'fathead': 'Wall decal poster life size player sports room',
    'mlb shop': 'Baseball jersey hat bat team logo official',
    'teepublic': 'Graphic tee printed shirt artist design fan',
    'personal creations': 'Personalized engraved custom name gift keepsake',
    'shutterfly': 'Photo book printed canvas holiday card picture',
    'free prints': 'Printed photo wallet size picture free shipping',
    'freeprints': 'Printed photo wallet size picture free shipping',
    'photo books': 'Hardcover photo album printed memories picture book',
    'ink cards': 'Greeting card holiday birthday printed custom',
    'bouqs': 'Fresh cut flower bouquet rose lily stem delivery',
    'yankee candle': 'Scented candle wax melt fragrance home decor',
    'laura geller': 'Foundation blush eyeshadow makeup cosmetics brush',
    'nogu': 'Beaded bracelet stacking jewelry minimalist handmade',
    'uber': 'Rideshare car ride request driver pickup drop off',
    'daily burn': 'Workout video streaming fitness class exercise routine',
    'ifit': 'Treadmill bike interactive trainer workout subscription',
    'accuweather': 'Weather forecast radar hourly temperature alert',
    'anaconda': 'Python coding data science environment package manager',
    'readmio': 'Childrens bedtime story reading interactive narrated',
    'magazine': 'Digital print publication subscription monthly issue',
    'rover': 'Dog walker pet sitter boarding overnight care',
    'crunch labs': 'STEM engineering build kit monthly box kids',
}

df_exploded['description'] = df_exploded['discount_item'].map(brand_descriptions)

In [ ]:
# Save to Drive as xlsx
df_exploded.to_excel("/content/drive/MyDrive/Stat_653/project/tmobiletuesdays.xlsx", index=False)
df_exploded

# Topic Modeling Using Latent Dirichlet Allocation (LDA)
### Step 1: Setup

Install the required Python libraries: pandas for data manipulation, gensim for topic modeling, spaCy for NLP processing, NLTK for stopwords, and matplotlib for visualization.

In [ ]:
!pip install pandas gensim spacy nltk matplotlib

Load the dataset from Google Drive.

In [ ]:
# data  = pd.read_excel('/content/drive/MyDrive/Stat_653/project/tmobiletuesdays.xlsx')
# data.head()

data = df_exploded.copy()
data.head()

Clean the article titles by removing extra whitespace, email addresses, apostrophes, and non-alphabetical characters, then convert to lowercase. This prepares the text for tokenization.

In [ ]:
def preprocess_text(text):
    if not isinstance(text, str):  # handle NaN
        return ''
    text = re.sub(r'\s+', ' ', text)          # Remove extra spaces
    text = re.sub(r'\S*@\S*\s?', '', text)    # Remove emails
    text = re.sub(r'\'', '', text)             # Remove apostrophes
    text = re.sub(r'[^a-zA-Z]', ' ', text)    # Remove non-alphabet characters
    text = text.lower()                        # Convert to lowercase
    return text

data['cleaned_description'] = data['description'].apply(preprocess_text)

Import gensim and NLTK, download the English stopwords list, then tokenize each cleaned title by splitting into individual words and removing common stopwords (e.g. 'the', 'and', 'is').

In [ ]:
import nltk
import gensim
from nltk.corpus import stopwords

# Download NLTK stopwords
nltk.download('stopwords')
stop_words = stopwords.words('english')

# Tokenize and remove stopwords
def tokenize(text):
    tokens = gensim.utils.simple_preprocess(text, deacc=True)
    tokens = [token for token in tokens if token not in stop_words]
    return tokens

data['tokens'] = data['cleaned_description'].apply(tokenize)

Load the spaCy English model and apply lemmatization to reduce each token to its base form (e.g. 'infections' → 'infection', 'testing' → 'test'). This reduces vocabulary size and improves topic quality.

In [ ]:
import spacy

# Load spaCy model
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

def lemmatize(tokens):
    doc = nlp(" ".join(tokens))
    return [token.lemma_ for token in doc]

data['lemmas'] = data['tokens'].apply(lemmatize)

Build the Gensim dictionary and corpus. The dictionary maps each unique word to an integer ID, and the corpus converts each document into a bag-of-words representation using those IDs.

In [ ]:
import gensim.corpora as corpora

# Create dictionary and corpus
id2word = corpora.Dictionary(data['lemmas'])
texts = data['lemmas']
corpus = [id2word.doc2bow(text) for text in texts]

Train the LDA (Latent Dirichlet Allocation) model with 3 topics on the corpus. Key parameters: 10 passes for thorough training, chunksize of 100 documents per batch, and alpha='auto' for automatic topic weight learning.

In [ ]:
# Build LDA model
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus,
                                            id2word=id2word,
                                            num_topics=7,
                                            random_state=100,
                                            update_every=1,
                                            chunksize=100,
                                            passes=10,
                                            alpha='auto',
                                            per_word_topics=True)

Print the top 10 words for each discovered topic along with their weights. Higher weights indicate the word is more representative of that topic.

In [ ]:
# Print the topics
topics = lda_model.print_topics(num_words=10)
for topic in topics:
    print(topic)

Define a function to assign the most probable topic to each document, map topic IDs to human-readable labels, and apply it to the dataset to create a new 'topic' column.

In [ ]:
def get_topic(text, dictionary, lda_model):
    # text is already a list of tokens, no need to split
    bow = dictionary.doc2bow(text)
    topics = lda_model.get_document_topics(bow)
    if topics:
        return max(topics, key=lambda x: x[1])[0]
    return -1

topic_labels = {0: 'Fried Chicken (Popeyes/KFC)',
                1: 'Pharmacy & Health',
                2: 'Gas & Fuel',
                3: 'Magazines & Subscriptions',
                4: 'Pizza (Little Caesars)',
                5: 'Fast Food (Burgers/Tacos)',
                6: 'Movies & Entertainment',
                -1: 'Unknown'}

# Now apply topics
data['topic_id'] = data['lemmas'].apply(lambda x: get_topic(x, id2word, lda_model))
data['topic'] = data['topic_id'].map(topic_labels)
data[['lemmas', 'topic']]

Plot a bar chart showing the percentage distribution of topics across the three topics to visualize how the model categorized the dataset.

In [ ]:
import matplotlib.pyplot as plt

topic_counts = data['topic'].value_counts(dropna=True)
total = len(data)
topic_pct = (topic_counts / total * 100).round(1)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22']

plt.figure(figsize=(12, 6))
bars = plt.bar(topic_counts.index.astype(str), topic_pct.values, color=colors[:len(topic_counts)])
plt.title('Topic Distribution')
plt.xlabel('Topic')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=20, ha='right')
for i, v in enumerate(topic_pct.values):
    plt.text(i, v + 0.5, f'{v}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

Install additional libraries needed for the Reddit scraping pipeline (commented out - used in development).

In [ ]:
import matplotlib.pyplot as plt

item_counts = data['discount_item'].value_counts()

plt.figure(figsize=(14, 6))
bars = plt.bar(item_counts.index.astype(str), item_counts.values, color='#1f77b4')
plt.title('Discount Item Frequency')
plt.xlabel('Brand')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
for i, v in enumerate(item_counts.values):
    plt.text(i, v + 0.5, str(v), ha='center', fontweight='bold', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

item_counts = data.groupby('discount_item')['count'].sum().sort_values(ascending=False)

plt.figure(figsize=(14, 6))
bars = plt.bar(item_counts.index.astype(str), item_counts.values, color='#1f77b4')
plt.title('Distributed Discounts by Item')
plt.xlabel('Brand')
plt.ylabel('Total Count')
plt.xticks(rotation=45, ha='right')
for i, v in enumerate(item_counts.values):
    plt.text(i, v + 0.5, str(v), ha='center', fontweight='bold', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
from gensim.models import CoherenceModel

# Compute coherence score
coherence_model_lda = CoherenceModel(model=lda_model, texts=texts, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('\nCoherence Score: ', coherence_lda)